# <b>Eksploracyjna analiza danych z uwzględnieniem czynnika przestrzennego</b>

<i>komórka robocza z uwagami w trakcie pracy; do usunięcia, gdy wszystko będzie gotowe</i><br>
<b style="color:red"> KOMÓRKI DO SPRAWDZENIA - są tam moje propozycje </b>
- z funkcjami do tworzenia modeli ML
- z głównym pipelinem

<b>Ogólnie</b>:
- proponuję lekką zmianę kodu, aby rozdzielić analizę i tworzenie modeli (modele na samym końcu notebooka przy podsumowaniu i opisie potencjalnych zastosowań w ML),
- spróbować wyeliminować duplikaty rzeczy, które zrobił każdy z nas (np. żeby 3 razy nie było liczenia dla krajów ilości wartości brakujących)
- proponuję na samym początku zrobić analizę globalną (ilość wartości brakujących dla zmiennych, i takie globalne statystyki całego zbioru), a dopiero potem robić analizę obszarów (sugeruję kolejność: regiony -> kraje -> miasta; schodzimy coraz niżej i uszczegóławiamy dane).
- trzeba do wykresów i tabel dopisać jasne interpretacje i dobrze, żeby to było zrobione na każdym poziomie analizy

Autorzy:
- Adam Dracz
- Filip Kandefer
- Rafał Szarowicz

## Opis danych
Analizowane dane dotyczą zanieczyszczeń powietrza i zostały udostępnione przez WHO.

Pochodzą one z pomiarów wykonywanych w wybranych miastach na całym świecie. Zmienne, które zawiera zbiór danych to:
- <b>Measurement Year</b>: Rok pomiaru.
- <b>PM2.5 (μg/m3)</b>: Stężenie pyłów PM2.5 w mikrogramach na metr sześcienny.
- <b>PM10 (μg/m3)</b>: Stężenie pyłów PM10 w mikrogramach na metr sześcienny.
- <b>NO2 (μg/m3)</b>: Stężenie dwutlenku azotu (NO2) w mikrogramach na metr sześcienny.
- <b>PM25 temporal coverage (%)</b>: Pokrycie czasowe danych dotyczących pyłów PM2.5 (%).
- <b>PM10 temporal coverage (%)</b>: Pokrycie czasowe danych dotyczących pyłów PM10 (%).
- <b>NO2 temporal coverage (%)</b>: Pokrycie czasowe danych dotyczących dwutlenku azotu (%).
- <b>Reference</b>: Odwołanie do źródła danych.
- <b>Number and type of monitoring stations</b>: Liczba i typ stacji monitorujących.
- <b>Version of the database</b>: Wersja bazy danych.
- <b>Status</b>: Status danych.

Przy czym stężenia pyłów i NO2 są roczną średnią, a pokrycie czasowe informuje w procentach o tym w jak dużej części roku dane były pobierane.

Analiza jest możliwa na trzech poziomach:
<ul>
    <li>miasta,</li>
    <li>kraje,</li>
    <li>regiony WHO: 
        <ul>
            <li>europejski,</li>
            <li>wschodni śródziemnomorski,</li>
            <li>amerykański,</li>
            <li>zachodni Pacyfiku,</li>
            <li>południowo-wschodniej Azji,</li>
            <li>afrykański</li>
        </ul></li>
</ul>

Pomiary dotyczą lat 2000 - 2021 (bez 2005).

Zgodnie z informacjami udostępnionymi przez WHO, dane te mają następujące ograniczenia, jeśli chodzi o porównanie między państwami:
- tylko część miast dla danego kraju jest ujęta,
- duża różnorodność technik i metod pomiarów,
- różne pokrycie czasowe danych, co skutkuje tym, że dane odbiegają od faktycznej rocznej średniej ze względu na uchwycenie zmienności sezonowej,
- różnorodność jakości pomiaru.

W związku z powyższym WHO zaleca wykorzystanie danych do tworzenia modeli, z wykorzystaniem innych zbiorów danych (np. z danymi satelitarnymi, dotyczącymi populacji, itd.) czy też uwidocznienia wzrastającej ilości stacji pomiarowych na świecie i zaznacza, że na podstawie danych nie można ustalić najbardziej zanieczyszczonych miast na świecie oraz niemożliwe jest bezpośrednie porównywanie krajów.
(<a href="https://www.who.int/data/gho/data/themes/air-pollution/who-air-quality-database">źródło</a>)

## Cel analizy
Celem jest przeprowadzenie eksploracyjnej analizy danych na trzech poziomach (miasta, kraje, regiony), próba znalezienia związków przyczynowo-skutkowych, wyjaśnienie braków w danych, próba ich uzupełnienia oraz przedstawienie potencjalnych zastosowań w uczeniu maszynowym.

## <b>Analiza danych</b>

In [ ]:
# =========================================================
# 1. IMPORTY I USTAWIENIA
# =========================================================
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
import ipywidgets as widgets

from IPython.display import display, Markdown, clear_output
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split

import geopandas as gpd


sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: f"{x:.3f}")

### Ustawienia

In [ ]:
# =========================================================
# 2. STAŁE I ŚCIEŻKI
# =========================================================

FILE_PATH = Path("who_aap_2021_v9_11august2022.xlsx")
SHEET_NAME = "AAP_2022_city_v9"

OUTPUT_DIR = Path("wyniki_eda")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

PM25_COL = "PM2.5 (μg/m3)"
PM10_COL = "PM10 (μg/m3)"
NO2_COL = "NO2 (μg/m3)"
YEAR_COL = "Measurement Year"
CITY_COL = "City or Locality"
COUNTRY_COL = "WHO Country Name"
REGION_COL = "WHO Region"

PM25_COV_COL = "PM25 temporal coverage (%)"
PM10_COV_COL = "PM10 temporal coverage (%)"
NO2_COV_COL = "NO2 temporal coverage (%)"

NUMERIC_COLS = [
    YEAR_COL,
    PM25_COL,
    PM10_COL,
    NO2_COL,
    PM25_COV_COL,
    PM10_COV_COL,
    NO2_COV_COL,
]

TREND_START_YEAR = 2010
TREND_END_YEAR = 2021
MIN_OBS_PER_GROUP_YEAR = 10


### Funkcje wykorzystywane w analizie

In [ ]:
# =========================================================
# 3. WCZYTANIE I PRZYGOTOWANIE DANYCH
# =========================================================

def load_data(file_path: Path, sheet_name: str) -> pd.DataFrame:
    df = pd.read_excel(file_path, sheet_name=sheet_name)
    df.columns = df.columns.str.strip()
    return df


def prepare_data(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    for col in NUMERIC_COLS:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")

    if "Status" in df.columns:
        df = df.drop(columns=["Status"])

    df = df.dropna(subset=[YEAR_COL, COUNTRY_COL, CITY_COL, REGION_COL]).copy()
    df[CITY_COL] = df[CITY_COL].astype(str).str.strip()
    df[COUNTRY_COL] = df[COUNTRY_COL].astype(str).str.strip()
    df[REGION_COL] = df[REGION_COL].astype(str).str.strip()

    df = df[df[CITY_COL] != ""].copy()

    return df


df_raw = load_data(FILE_PATH, SHEET_NAME)
df = prepare_data(df_raw)

print("Kształt danych:", df.shape)
display(df.head())


In [ ]:
# =========================================================
# 4. ANALIZA BRAKÓW
# =========================================================

def get_missing_summary(df: pd.DataFrame) -> pd.DataFrame:
    return pd.DataFrame({
        "braki": df.isna().sum(),
        "procent_brakow": (df.isna().mean() * 100).round(2)
    }).sort_values("procent_brakow", ascending=False)


def get_missing_by_group(df: pd.DataFrame, group_col: str) -> pd.DataFrame:
    cols = [PM25_COL, PM10_COL, NO2_COL]
    return (
        df.groupby(group_col)[cols]
        .apply(lambda x: x.isna().mean() * 100)
        .round(2)
        .sort_index()
    )


missing_summary = get_missing_summary(df)
missing_by_region = get_missing_by_group(df, REGION_COL)
missing_by_country = get_missing_by_group(df, COUNTRY_COL)

display(missing_summary)
display(missing_by_region.head(20))

In [ ]:
# =========================================================
# 5. IMPUTACJA PM2.5
# =========================================================

def calculate_ratio_by_region(df: pd.DataFrame) -> tuple[pd.DataFrame, pd.Series]:
    ratio_df = df[df[PM25_COL].notna() & df[PM10_COL].notna()].copy()
    ratio_df["pm25_pm10_ratio"] = ratio_df[PM25_COL] / ratio_df[PM10_COL]

    ratio_df = ratio_df[
        (ratio_df["pm25_pm10_ratio"] > 0) &
        (ratio_df["pm25_pm10_ratio"] <= 1.5)
    ].copy()

    ratio_by_region = ratio_df.groupby(REGION_COL)["pm25_pm10_ratio"].median()
    return ratio_df, ratio_by_region


def impute_pm25(df: pd.DataFrame, ratio_by_region: pd.Series) -> pd.DataFrame:
    df = df.copy()

    df["PM2.5_was_imputed"] = df[PM25_COL].isna()
    df["PM2.5_imputed"] = df[PM25_COL]

    mask_ratio = df["PM2.5_imputed"].isna() & df[PM10_COL].notna()
    df.loc[mask_ratio, "PM2.5_imputed"] = (
        df.loc[mask_ratio, PM10_COL] *
        df.loc[mask_ratio, REGION_COL].map(ratio_by_region)
    )

    df["PM2.5_imputed"] = (
        df.groupby([REGION_COL, YEAR_COL])["PM2.5_imputed"]
        .transform(lambda x: x.fillna(x.median()))
    )

    df["PM2.5_imputed"] = (
        df.groupby(REGION_COL)["PM2.5_imputed"]
        .transform(lambda x: x.fillna(x.median()))
    )

    df["PM2.5_imputed"] = df["PM2.5_imputed"].fillna(df["PM2.5_imputed"].median())

    return df


def get_imputation_summary(df_before: pd.DataFrame, df_after: pd.DataFrame) -> pd.DataFrame:
    return pd.DataFrame({
        "braki_przed": [df_before[PM25_COL].isna().sum()],
        "braki_po": [df_after["PM2.5_imputed"].isna().sum()],
        "udzial_imputowanych_%": [round(df_after["PM2.5_was_imputed"].mean() * 100, 2)]
    })


def get_imputation_share_by_group(df: pd.DataFrame, group_col: str) -> pd.DataFrame:
    return (
        df.groupby(group_col)["PM2.5_was_imputed"]
        .mean()
        .mul(100)
        .round(2)
        .sort_values(ascending=False)
        .to_frame("udzial_imputowanych_%")
    )


ratio_df, ratio_by_region = calculate_ratio_by_region(df)
df = impute_pm25(df, ratio_by_region)

imputation_summary = get_imputation_summary(df_raw if PM25_COL in df_raw.columns else df, df)
imputation_by_region = get_imputation_share_by_group(df, REGION_COL)

display(ratio_by_region.to_frame("median_pm25_pm10_ratio"))
display(imputation_summary)
display(imputation_by_region)

In [ ]:
# =========================================================
# 6. FUNKCJE AGREGUJĄCE WSPÓLNE
# =========================================================

def aggregate_level(df: pd.DataFrame, level_col: str) -> pd.DataFrame:
    return (
        df.groupby(level_col)
        .agg(
            pm25_mean=("PM2.5_imputed", "mean"),
            pm25_median=("PM2.5_imputed", "median"),
            pm25_std=("PM2.5_imputed", "std"),
            pm10_mean=(PM10_COL, "mean"),
            no2_mean=(NO2_COL, "mean"),
            n_obs=("PM2.5_imputed", "size"),
            n_countries=(COUNTRY_COL, "nunique"),
            n_cities=(CITY_COL, "nunique"),
        )
        .sort_values("pm25_mean", ascending=False)
    )


def get_level_year_stats(df: pd.DataFrame, level_col: str) -> pd.DataFrame:
    return (
        df.groupby([level_col, YEAR_COL])
        .agg(
            pm25_mean=("PM2.5_imputed", "mean"),
            pm25_median=("PM2.5_imputed", "median"),
            pm25_std=("PM2.5_imputed", "std"),
            n_obs=("PM2.5_imputed", "size"),
        )
        .reset_index()
        .sort_values([level_col, YEAR_COL])
    )


def filter_level_year_stats(
    stats_df: pd.DataFrame,
    start_year: int = TREND_START_YEAR,
    end_year: int = TREND_END_YEAR,
    min_obs: int = MIN_OBS_PER_GROUP_YEAR
) -> pd.DataFrame:
    return stats_df[
        (stats_df[YEAR_COL] >= start_year) &
        (stats_df[YEAR_COL] <= end_year) &
        (stats_df["n_obs"] >= min_obs)
    ].copy()


def compare_before_after_imputation(df: pd.DataFrame, level_col: str) -> pd.DataFrame:
    comparison = (
        df.groupby(level_col)
        .agg(
            pm25_mean_before=(PM25_COL, "mean"),
            pm25_mean_after=("PM2.5_imputed", "mean"),
            pm25_median_before=(PM25_COL, "median"),
            pm25_median_after=("PM2.5_imputed", "median"),
            imputacja_udzial=("PM2.5_was_imputed", lambda x: x.mean() * 100),
        )
        .round(3)
    )

    comparison["roznica_mean"] = (
        comparison["pm25_mean_after"] - comparison["pm25_mean_before"]
    ).round(3)

    comparison["roznica_median"] = (
        comparison["pm25_median_after"] - comparison["pm25_median_before"]
    ).round(3)

    return comparison.sort_values("pm25_mean_after", ascending=False)

In [ ]:
# =========================================================
# 7. FUNKCJE WYKRESÓW WSPÓLNE
# =========================================================

def plot_distribution(df: pd.DataFrame, value_col: str, title: str, filename: str | None = None) -> None:
    plt.figure(figsize=(10, 5))
    sns.histplot(df[value_col].dropna(), bins=40)
    plt.title(title)
    plt.xlabel(value_col)
    plt.ylabel("Liczba obserwacji")
    plt.tight_layout()
    if filename:
        plt.savefig(OUTPUT_DIR / filename, dpi=200)
    plt.show()


def plot_ranking(summary_df: pd.DataFrame, index_name: str, metric: str, title: str, filename: str | None = None) -> None:
    plt.figure(figsize=(11, 6))
    sns.barplot(
        data=summary_df.reset_index(),
        x=metric,
        y=index_name
    )
    plt.title(title)
    plt.xlabel(metric)
    plt.ylabel(index_name)
    plt.tight_layout()
    if filename:
        plt.savefig(OUTPUT_DIR / filename, dpi=200)
    plt.show()


def plot_boxplot(df: pd.DataFrame, group_col: str, group_order: list[str], title: str, filename: str | None = None, show_fliers: bool = False) -> None:
    plt.figure(figsize=(12, 6))
    sns.boxplot(
        data=df,
        x=group_col,
        y="PM2.5_imputed",
        order=group_order,
        showfliers=show_fliers
    )
    plt.title(title)
    plt.xlabel(group_col)
    plt.ylabel("PM2.5_imputed")
    plt.xticks(rotation=40, ha="right")
    plt.tight_layout()
    if filename:
        plt.savefig(OUTPUT_DIR / filename, dpi=200)
    plt.show()


def plot_trend(stats_df: pd.DataFrame, group_col: str, value_col: str, title: str, filename: str | None = None) -> None:
    plt.figure(figsize=(12, 6))
    sns.lineplot(
        data=stats_df,
        x=YEAR_COL,
        y=value_col,
        hue=group_col,
        marker="o"
    )
    plt.title(title)
    plt.xlabel("Rok")
    plt.ylabel(value_col)
    plt.tight_layout()
    if filename:
        plt.savefig(OUTPUT_DIR / filename, dpi=200)
    plt.show()


def plot_counts(stats_df: pd.DataFrame, group_col: str, title: str, filename: str | None = None) -> None:
    plt.figure(figsize=(12, 6))
    sns.lineplot(
        data=stats_df,
        x=YEAR_COL,
        y="n_obs",
        hue=group_col,
        marker="o"
    )
    plt.title(title)
    plt.xlabel("Rok")
    plt.ylabel("Liczba obserwacji")
    plt.tight_layout()
    if filename:
        plt.savefig(OUTPUT_DIR / filename, dpi=200)
    plt.show()


def plot_corr_heatmap(df: pd.DataFrame, title: str, filename: str | None = None) -> pd.DataFrame:
    corr_cols = ["PM2.5_imputed", PM10_COL, NO2_COL]
    corr = df[corr_cols].corr()

    plt.figure(figsize=(6, 5))
    sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0)
    plt.title(title)
    plt.tight_layout()
    if filename:
        plt.savefig(OUTPUT_DIR / filename, dpi=200)
    plt.show()

    return corr


def plot_before_after_comparison(comparison_df: pd.DataFrame, level_col: str, title: str, filename: str | None = None) -> None:
    temp = comparison_df.reset_index()[[level_col, "pm25_mean_before", "pm25_mean_after"]].melt(
        id_vars=level_col,
        var_name="wariant",
        value_name="PM2.5"
    )

    plt.figure(figsize=(11, 6))
    sns.barplot(data=temp, x="PM2.5", y=level_col, hue="wariant")
    plt.title(title)
    plt.xlabel("Średnie PM2.5")
    plt.ylabel(level_col)
    plt.tight_layout()
    if filename:
        plt.savefig(OUTPUT_DIR / filename, dpi=200)
    plt.show()


In [ ]:
# =========================================================
# 8. PODSUMOWANIE OGÓLNE ZBIORU
# =========================================================

def print_basic_summary(df: pd.DataFrame) -> None:
    print("=" * 60)
    print("PODSUMOWANIE ZBIORU")
    print("=" * 60)
    print(f"Liczba rekordów: {len(df):,}")
    print(f"Liczba regionów: {df[REGION_COL].nunique():,}")
    print(f"Liczba krajów: {df[COUNTRY_COL].nunique():,}")
    print(f"Liczba miast: {df[CITY_COL].nunique():,}")
    print(f"Zakres lat: {int(df[YEAR_COL].min())} - {int(df[YEAR_COL].max())}")
    print("=" * 60)


print_basic_summary(df)

In [ ]:
# =========================================================
# 9. ANALIZA REGIONÓW: TABELE
# =========================================================

region_summary = aggregate_level(df, REGION_COL)
region_order = region_summary.index.tolist()

region_year_stats = get_level_year_stats(df, REGION_COL)
region_year_stats_filtered = filter_level_year_stats(region_year_stats)

region_before_after = compare_before_after_imputation(df, REGION_COL)

display(region_summary)
display(region_year_stats_filtered.head(20))
display(region_before_after)

In [ ]:
# =========================================================
# 10. ANALIZA REGIONÓW: WYKRESY
# =========================================================

plot_distribution(
    df,
    value_col="PM2.5_imputed",
    title="Rozkład PM2.5 po imputacji - cały zbiór",
    filename="01_hist_pm25_all.png"
)

plot_ranking(
    region_summary,
    index_name=REGION_COL,
    metric="pm25_mean",
    title="Ranking regionów WHO według średniego PM2.5",
    filename="02_region_ranking_mean.png"
)

plot_ranking(
    region_summary.sort_values("pm25_median", ascending=False),
    index_name=REGION_COL,
    metric="pm25_median",
    title="Ranking regionów WHO według mediany PM2.5",
    filename="03_region_ranking_median.png"
)

plot_boxplot(
    df,
    group_col=REGION_COL,
    group_order=region_order,
    title="Rozkład PM2.5 w regionach WHO",
    filename="04_region_boxplot.png",
    show_fliers=False
)

plot_trend(
    region_year_stats_filtered,
    group_col=REGION_COL,
    value_col="pm25_mean",
    title=f"Trend średniego PM2.5 w regionach ({TREND_START_YEAR}-{TREND_END_YEAR})",
    filename="05_region_trend_mean.png"
)

plot_trend(
    region_year_stats_filtered,
    group_col=REGION_COL,
    value_col="pm25_median",
    title=f"Trend mediany PM2.5 w regionach ({TREND_START_YEAR}-{TREND_END_YEAR})",
    filename="06_region_trend_median.png"
)

plot_counts(
    region_year_stats_filtered,
    group_col=REGION_COL,
    title="Liczba obserwacji w czasie według regionów",
    filename="07_region_counts.png"
)

plot_before_after_comparison(
    region_before_after,
    level_col=REGION_COL,
    title="Porównanie średniego PM2.5 przed i po imputacji - regiony",
    filename="08_region_before_after.png"
)

corr_all = plot_corr_heatmap(
    df,
    title="Macierz współczynników korelacji Pearsona zanieczyszczeń - cały zbiór",
    filename="09_corr_all.png"
)

In [ ]:
# =========================================================
# 11. ANALIZA KRAJÓW: TABELE
# =========================================================

country_summary = aggregate_level(df, COUNTRY_COL)
country_year_stats = get_level_year_stats(df, COUNTRY_COL)
country_year_stats_filtered = filter_level_year_stats(country_year_stats)

country_before_after = compare_before_after_imputation(df, COUNTRY_COL)

display(country_summary.head(20))
display(country_year_stats_filtered.head(20))
display(country_before_after.head(20))


In [ ]:
# =========================================================
# 12. ANALIZA KRAJÓW: WYKRESY GŁÓWNE
# =========================================================

top_n_countries = 15
top_countries = country_summary.head(top_n_countries).index.tolist()

plot_ranking(
    country_summary.head(top_n_countries),
    index_name=COUNTRY_COL,
    metric="pm25_mean",
    title=f"Top {top_n_countries} krajów wg średniego PM2.5",
    filename="10_country_ranking_mean.png"
)

plot_ranking(
    country_summary.sort_values("pm25_median", ascending=False).head(top_n_countries),
    index_name=COUNTRY_COL,
    metric="pm25_median",
    title=f"Top {top_n_countries} krajów wg mediany PM2.5",
    filename="11_country_ranking_median.png"
)

country_map_df = (
    df.groupby(COUNTRY_COL)
    .agg(
        pm25_mean=("PM2.5_imputed", "mean"),
        n_obs=("PM2.5_imputed", "size")
    )
    .reset_index()
)

fig = px.choropleth(
    country_map_df,
    locations=COUNTRY_COL,
    locationmode="country names",
    color="pm25_mean",
    hover_data=["n_obs"],
    color_continuous_scale="Reds",
    title="Średnie PM2.5 według krajów"
)
fig.show()


In [ ]:
# =========================================================
# 13. ANALIZA KRAJÓW: KRAJE Z NAJWIĘKSZĄ MEDIANĄ W KOLEJNYCH LATACH
# =========================================================

years = sorted(df[YEAR_COL].dropna().unique())

pollution_cols = pd.MultiIndex.from_product(
    [["PM2.5", "PM10", "NO2"], ["Country", "Median"]]
)

biggest_polluters = pd.DataFrame(index=years, columns=pollution_cols)
biggest_polluters.index.name = "Year"

for year in years:
    temp = df[df[YEAR_COL] == year].groupby(COUNTRY_COL)[[PM25_COL, PM10_COL, NO2_COL]].median()
    temp["Country"] = temp.index

    if temp[PM25_COL].notna().any():
        row = temp.sort_values(PM25_COL, ascending=False).iloc[0]
        biggest_polluters.loc[year, ("PM2.5", "Country")] = row["Country"]
        biggest_polluters.loc[year, ("PM2.5", "Median")] = row[PM25_COL]

    if temp[PM10_COL].notna().any():
        row = temp.sort_values(PM10_COL, ascending=False).iloc[0]
        biggest_polluters.loc[year, ("PM10", "Country")] = row["Country"]
        biggest_polluters.loc[year, ("PM10", "Median")] = row[PM10_COL]

    if temp[NO2_COL].notna().any():
        row = temp.sort_values(NO2_COL, ascending=False).iloc[0]
        biggest_polluters.loc[year, ("NO2", "Country")] = row["Country"]
        biggest_polluters.loc[year, ("NO2", "Median")] = row[NO2_COL]

display(biggest_polluters)

In [ ]:
# =========================================================
# 14. ANALIZA KRAJÓW: PRZYGOTOWANIE MAP I FUNKCJE POMOCNICZE
# =========================================================
WORLD_FILE = Path("ne_10m_admin_0_countries.zip")

if not WORLD_FILE.exists():
    raise FileNotFoundError(
        f"Nie znaleziono pliku z mapą świata: {WORLD_FILE}. "
        "Upewnij się, że plik ne_10m_admin_0_countries.zip znajduje się w folderze projektu."
    )

world = gpd.read_file(WORLD_FILE)

required_world_cols = ["ADM0_A3", "geometry"]
missing_world_cols = [col for col in required_world_cols if col not in world.columns]
if missing_world_cols:
    raise ValueError(f"W pliku mapy świata brakuje kolumn: {missing_world_cols}")

world = world[required_world_cols].copy()

if "ISO3" not in df.columns:
    raise ValueError("W danych brakuje kolumny 'ISO3', potrzebnej do łączenia danych z mapą świata.")

sea = gpd.GeoSeries([world.geometry.union_all().envelope], crs=world.crs)
sea_coords = list(sea.iloc[0].exterior.coords)

xmin = min(x for x, y in sea_coords)
xmax = max(x for x, y in sea_coords)
ymin = min(y for x, y in sea_coords)
ymax = max(y for x, y in sea_coords)


def get_country_agg_for_map(
    df: pd.DataFrame,
    var: str,
    stat: str = "mean",
    year: int | str = "all"
) -> gpd.GeoDataFrame:
    """
    Przygotowuje GeoDataFrame do mapowania krajów.
    stat:
        - 'mean'
        - 'median'
        - 'nan_counts'
        - 'city_counts'
    year:
        - konkretny rok lub 'all'
    """
    if year == "all":
        data = df.copy()
    else:
        data = df[df[YEAR_COL] == year].copy()

    if stat in {"mean", "median"}:
        grouped = (
            data.groupby("ISO3")[var]
            .agg(stat)
            .reset_index(name="value")
        )
    elif stat == "nan_counts":
        grouped = (
            data.groupby("ISO3")[var]
            .apply(lambda x: x.isna().sum())
            .reset_index(name="value")
        )
    elif stat == "city_counts":
        grouped = (
            data.groupby("ISO3")[CITY_COL]
            .nunique()
            .reset_index(name="value")
        )
    else:
        raise ValueError("stat musi być jednym z: 'mean', 'median', 'nan_counts', 'city_counts'")

    geo_df = world.merge(grouped, left_on="ADM0_A3", right_on="ISO3", how="left")
    return gpd.GeoDataFrame(geo_df, geometry="geometry", crs=world.crs)


def plot_country_map(
    df: pd.DataFrame,
    var: str,
    stat: str = "mean",
    year: int | str = "all",
    cmap: str | None = None,
    title: str | None = None,
    filename: str | None = None
) -> None:
    cmap_map = {
        PM25_COL: "Purples",
        PM10_COL: "Reds",
        NO2_COL: "Greens",
        "city_counts": "Oranges",
        "nan_counts": "Greys"
    }

    geo_df = get_country_agg_for_map(df, var=var, stat=stat, year=year)

    if title is None:
        year_txt = "dla wszystkich lat" if year == "all" else f"dla roku {year}"
        if stat == "city_counts":
            title = f"Liczba miast przypisanych do krajów {year_txt}"
        elif stat == "nan_counts":
            title = f"Liczba braków danych ({var}) {year_txt}"
        else:
            title = f"{stat.capitalize()} {var} według krajów {year_txt}"

    if cmap is None:
        if stat == "city_counts":
            cmap = cmap_map["city_counts"]
        elif stat == "nan_counts":
            cmap = cmap_map["nan_counts"]
        else:
            cmap = cmap_map.get(var, "viridis")

    fig, ax = plt.subplots(figsize=(14, 8))

    sea.plot(ax=ax, color="azure")
    geo_df.plot(
        column="value",
        ax=ax,
        cmap=cmap,
        legend=True,
        missing_kwds={"color": "lightgrey", "label": "Brak danych"}
    )
    geo_df.boundary.plot(ax=ax, linewidth=0.5, color="black")

    ax.set_xlim(xmin, xmax)
    ax.set_ylim(ymin, ymax)
    ax.set_xlabel("Długość geograficzna")
    ax.set_ylabel("Szerokość geograficzna")
    ax.set_title(title)

    plt.tight_layout()
    if filename:
        plt.savefig(OUTPUT_DIR / filename, dpi=200, bbox_inches="tight")
    plt.show()


def plot_country_violin(
    df: pd.DataFrame,
    value_col: str,
    title: str,
    filename: str | None = None
) -> None:
    plot_df = df[[YEAR_COL, value_col]].dropna().copy()
    plot_df[YEAR_COL] = plot_df[YEAR_COL].astype(int)

    plt.figure(figsize=(14, 6))
    sns.violinplot(
        data=plot_df,
        x=YEAR_COL,
        y=value_col,
        hue=YEAR_COL,
        legend=False,
        palette="Set2"
    )
    plt.title(title)
    plt.xlabel("Rok")
    plt.ylabel(value_col)
    plt.xticks(rotation=45)
    plt.tight_layout()
    if filename:
        plt.savefig(OUTPUT_DIR / filename, dpi=200)
    plt.show()

In [ ]:
# =========================================================
# 15. ANALIZA KRAJÓW: MAPY ŚREDNICH I MEDIAN
# =========================================================

plot_country_map(
    df,
    var=PM25_COL,
    stat="mean",
    year="all",
    title="Średnie PM2.5 według krajów - wszystkie lata",
    filename="12_country_map_pm25_mean_all.png"
)

plot_country_map(
    df,
    var=PM25_COL,
    stat="median",
    year="all",
    title="Mediana PM2.5 według krajów - wszystkie lata",
    filename="13_country_map_pm25_median_all.png"
)

plot_country_map(
    df,
    var=PM10_COL,
    stat="mean",
    year="all",
    title="Średnie PM10 według krajów - wszystkie lata",
    filename="14_country_map_pm10_mean_all.png"
)

plot_country_map(
    df,
    var=PM10_COL,
    stat="median",
    year="all",
    title="Mediana PM10 według krajów - wszystkie lata",
    filename="15_country_map_pm10_median_all.png"
)

plot_country_map(
    df,
    var=NO2_COL,
    stat="mean",
    year="all",
    title="Średnie NO2 według krajów - wszystkie lata",
    filename="16_country_map_no2_mean_all.png"
)

plot_country_map(
    df,
    var=NO2_COL,
    stat="median",
    year="all",
    title="Mediana NO2 według krajów - wszystkie lata",
    filename="17_country_map_no2_median_all.png"
)

In [ ]:
# =========================================================
# 16. ANALIZA KRAJÓW: MAPA DLA WYBRANEGO ROKU + PODGLĄD 2021
# =========================================================

selected_year = 2021

if selected_year in df[YEAR_COL].dropna().unique():
    plot_country_map(
        df,
        var=PM25_COL,
        stat="median",
        year=selected_year,
        title=f"Mediana PM2.5 według krajów - rok {selected_year}",
        filename="18_country_map_pm25_median_2021.png"
    )

    year_2021_df = (
        df[df[YEAR_COL] == selected_year]
        [[YEAR_COL, COUNTRY_COL, CITY_COL, REGION_COL, PM25_COL, PM10_COL, NO2_COL]]
        .sort_values([COUNTRY_COL, CITY_COL])
    )

    display(year_2021_df.head(50))
    print(f"Liczba rekordów dla {selected_year}: {len(year_2021_df)}")
    print(f"Liczba krajów dla {selected_year}: {year_2021_df[COUNTRY_COL].nunique()}")
    print(f"Liczba miast dla {selected_year}: {year_2021_df[CITY_COL].nunique()}")
else:
    print(f"Brak danych dla roku {selected_year}.")

In [ ]:
# =========================================================
# 17. ANALIZA KRAJÓW: VIOLINPLOTY PO LATACH
# =========================================================

plot_country_violin(
    df,
    value_col=PM25_COL,
    title="Violinplot PM2.5 w zależności od roku",
    filename="19_country_violin_pm25_by_year.png"
)

plot_country_violin(
    df,
    value_col=PM10_COL,
    title="Violinplot PM10 w zależności od roku",
    filename="20_country_violin_pm10_by_year.png"
)

plot_country_violin(
    df,
    value_col=NO2_COL,
    title="Violinplot NO2 w zależności od roku",
    filename="21_country_violin_no2_by_year.png"
)

In [ ]:
# =========================================================
# 18. ANALIZA KRAJÓW: CO SIĘ DZIEJE PRZED 2010?
# =========================================================

pre_2010 = df[df[YEAR_COL] < 2010].copy()

print("Liczba rekordów przed 2010 rokiem:", len(pre_2010))
print("Kraje występujące przed 2010 rokiem:")
display(pd.Series(sorted(pre_2010[COUNTRY_COL].dropna().unique()), name=COUNTRY_COL))

print("\nPodgląd danych przed 2010 rokiem:")
display(
    pre_2010[
        [YEAR_COL, COUNTRY_COL, CITY_COL, PM25_COL, PM10_COL, NO2_COL]
    ].sort_values([YEAR_COL, COUNTRY_COL, CITY_COL]).head(100)
)

print("\nTabela największych median od 2010 roku:")
display(biggest_polluters[biggest_polluters.index >= 2010])

In [ ]:
# =========================================================
# 19. ANALIZA KRAJÓW: Z ILU MIAST POCHODZĄ DANE?
# =========================================================

cities_per_country = (
    df.groupby([COUNTRY_COL, "ISO3"])[CITY_COL]
    .nunique()
    .reset_index(name="n_cities")
    .sort_values("n_cities", ascending=False)
)

display(cities_per_country.head(30))

plot_country_map(
    df,
    var=PM25_COL,   
    stat="city_counts",
    year="all",
    title="Liczba miast, z których pochodzą dane dla krajów",
    filename="22_country_map_city_counts.png"
)

In [ ]:
# =========================================================
# 20. ANALIZA KRAJÓW: BRAKI DANYCH I MAPA BRAKÓW
# =========================================================

missing_by_year = (
    df.groupby(YEAR_COL)[[PM25_COL, PM10_COL, NO2_COL]]
    .apply(lambda x: x.isna().sum())
)

non_missing_by_year = (
    df.groupby(YEAR_COL)[[PM25_COL, PM10_COL, NO2_COL]]
    .apply(lambda x: x.notna().sum())
)

print("Liczba braków danych po latach:")
display(missing_by_year)

print("Liczba niebrakujących wartości po latach:")
display(non_missing_by_year)

country_missing_percent = (
    df.groupby(COUNTRY_COL)[[PM25_COL, PM10_COL, NO2_COL]]
    .apply(lambda x: x.isna().mean() * 100)
    .round(2)
    .sort_values(by=PM25_COL, ascending=False)
)

print("Procent braków danych po krajach:")
display(country_missing_percent.head(30))

plot_country_map(
    df,
    var=PM10_COL,
    stat="nan_counts",
    year="all",
    title="Liczba braków danych PM10 według krajów",
    filename="23_country_map_missing_pm10.png"
)

In [ ]:
# =========================================================
# 21. ANALIZA KRAJÓW: POKRYCIE CZASOWE
# =========================================================

coverage_summary = pd.DataFrame({
    "zmienna": [PM25_COV_COL, PM10_COV_COL, NO2_COV_COL],
    "liczba_niebrakujacych": [
        df[PM25_COV_COL].notna().sum(),
        df[PM10_COV_COL].notna().sum(),
        df[NO2_COV_COL].notna().sum(),
    ],
    "udzial_coverage_<=50%": [
        round((df[PM25_COV_COL].le(50) & df[PM25_COV_COL].notna()).mean() * 100, 2),
        round((df[PM10_COV_COL].le(50) & df[PM10_COV_COL].notna()).mean() * 100, 2),
        round((df[NO2_COV_COL].le(50) & df[NO2_COV_COL].notna()).mean() * 100, 2),
    ],
    "procent_brakow": [
        round(df[PM25_COV_COL].isna().mean() * 100, 2),
        round(df[PM10_COV_COL].isna().mean() * 100, 2),
        round(df[NO2_COV_COL].isna().mean() * 100, 2),
    ]
})

display(coverage_summary)

coverage_by_country = (
    df.groupby(COUNTRY_COL)[[PM25_COV_COL, PM10_COV_COL, NO2_COV_COL]]
    .agg(["mean", "median", "count"])
)

display(coverage_by_country.head(20))

print(
    "Uwaga interpretacyjna: niskie pokrycie czasowe oznacza, że część pomiarów "
    "może nie reprezentować całego roku. Dodatkowo duży odsetek braków w kolumnach "
    "temporal coverage ogranicza możliwość pełnych porównań między krajami."
)

In [ ]:
# =========================================================
# 22. ANALIZA MIAST: TABELE
# =========================================================

city_summary = aggregate_level(df, CITY_COL)
city_year_stats = get_level_year_stats(df, CITY_COL)
city_year_stats_filtered = filter_level_year_stats(city_year_stats)

city_before_after = compare_before_after_imputation(df, CITY_COL)

display(city_summary.head(20))
display(city_year_stats_filtered.head(20))
display(city_before_after.head(20))

In [ ]:
# =========================================================
# 23. ANALIZA MIAST: WYKRESY GŁÓWNE
# =========================================================

top_n_cities = 15

plot_ranking(
    city_summary.head(top_n_cities),
    index_name=CITY_COL,
    metric="pm25_mean",
    title=f"Top {top_n_cities} miast wg średniego PM2.5",
    filename="12_city_ranking_mean.png"
)

plot_ranking(
    city_summary.sort_values("pm25_median", ascending=False).head(top_n_cities),
    index_name=CITY_COL,
    metric="pm25_median",
    title=f"Top {top_n_cities} miast wg mediany PM2.5",
    filename="13_city_ranking_median.png"
)

yearly_city_global = df.groupby(YEAR_COL)["PM2.5_imputed"].agg(["mean", "std", "count"]).dropna()

plt.figure(figsize=(11, 5))
plt.plot(yearly_city_global.index, yearly_city_global["mean"], marker="o", linewidth=2)
plt.fill_between(
    yearly_city_global.index,
    yearly_city_global["mean"] - yearly_city_global["std"].fillna(0),
    yearly_city_global["mean"] + yearly_city_global["std"].fillna(0),
    alpha=0.2
)
plt.title("Trend PM2.5 w czasie - poziom miast")
plt.xlabel("Rok")
plt.ylabel("PM2.5")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "14_city_global_trend.png", dpi=200)
plt.show()

In [ ]:
# =========================================================
# 24. ZALEŻNOŚCI MIĘDZY ZANIECZYSZCZENIAMI DLA WYBRANYCH KRAJÓW 
# =========================================================

def plot_for_country(country_name: str) -> None:
    country_data = df[df[COUNTRY_COL] == country_name].copy()

    rep_no2 = country_data[country_data["PM2.5_imputed"].notna() & country_data[NO2_COL].notna()]
    rep_pm10 = country_data[country_data["PM2.5_imputed"].notna() & country_data[PM10_COL].notna()]

    fig, axs = plt.subplots(1, 2, figsize=(12, 5), tight_layout=True)
    fig.suptitle(f"Zależności między zanieczyszczeniami - {country_name}", fontsize=14, fontweight="bold")

    sns.scatterplot(data=rep_no2, x="PM2.5_imputed", y=NO2_COL, ax=axs[0])
    axs[0].set_title("PM2.5 vs NO2")

    sns.scatterplot(data=rep_pm10, x="PM2.5_imputed", y=PM10_COL, ax=axs[1])
    axs[1].set_title("PM2.5 vs PM10")

    plt.show()


for c in ["China", "India", "Italy", "United States of America"]:
    if c in df[COUNTRY_COL].unique():
        plot_for_country(c)

In [ ]:
# =========================================================
# 25. MODEL MACHINE LEARNING 
# =========================================================

def prepare_ml_data(df: pd.DataFrame) -> tuple[pd.DataFrame, list[str]]:
    ml_df = df.copy()

    numeric_features = [
        PM10_COL,
        NO2_COL,
        PM25_COV_COL,
        PM10_COV_COL,
        NO2_COV_COL,
        YEAR_COL,
    ]
    numeric_features = [col for col in numeric_features if col in ml_df.columns]

    categorical_features = [REGION_COL]

    selected_cols = ["PM2.5_imputed"] + numeric_features + categorical_features
    ml_df = ml_df[selected_cols].dropna().copy()

    ml_df = pd.get_dummies(ml_df, columns=categorical_features, drop_first=True)
    feature_cols = [col for col in ml_df.columns if col != "PM2.5_imputed"]

    return ml_df, feature_cols


def run_ml_model(
    ml_df: pd.DataFrame,
    feature_cols: list[str],
    target_col: str = "PM2.5_imputed"
) -> dict:
    X = ml_df[feature_cols]
    y = ml_df[target_col]

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42
    )

    model = RandomForestRegressor(
        n_estimators=300,
        max_depth=18,
        min_samples_leaf=3,
        random_state=42,
        n_jobs=-1
    )
    model.fit(X_train, y_train)

    y_train_pred = model.predict(X_train)
    y_test_pred = model.predict(X_test)

    metrics = {
        "r2_train": r2_score(y_train, y_train_pred),
        "r2_test": r2_score(y_test, y_test_pred),
        "rmse_train": float(np.sqrt(mean_squared_error(y_train, y_train_pred))),
        "rmse_test": float(np.sqrt(mean_squared_error(y_test, y_test_pred))),
        "mae_test": float(mean_absolute_error(y_test, y_test_pred)),
        "n_samples": len(ml_df),
        "n_features": len(feature_cols),
    }

    importance = (
        pd.DataFrame({
            "feature": feature_cols,
            "importance": model.feature_importances_
        })
        .sort_values("importance", ascending=False)
        .reset_index(drop=True)
    )

    return {
        "model": model,
        "metrics": metrics,
        "importance": importance,
        "X_test": X_test,
        "y_test": y_test,
        "y_test_pred": y_test_pred,
    }


ml_df, ml_features = prepare_ml_data(df)
model_result = run_ml_model(ml_df, ml_features)

display(pd.DataFrame([model_result["metrics"]]))
display(model_result["importance"].head(15))

In [ ]:
# =========================================================
# 26. WYKRESY MACHINE LEARNING 
# =========================================================

importance = model_result["importance"]
y_test = model_result["y_test"]
y_test_pred = model_result["y_test_pred"]
metrics = model_result["metrics"]

plt.figure(figsize=(10, 6))
sns.barplot(data=importance.head(12), x="importance", y="feature")
plt.title("Najważniejsze cechy w modelu PM2.5")
plt.xlabel("Ważność cechy")
plt.ylabel("Cecha")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "15_ml_feature_importance.png", dpi=200)
plt.show()

plt.figure(figsize=(7, 7))
plt.scatter(y_test, y_test_pred, alpha=0.35)
lims = [min(y_test.min(), y_test_pred.min()), max(y_test.max(), y_test_pred.max())]
plt.plot(lims, lims, "r--", linewidth=2)
plt.xlabel("Rzeczywiste PM2.5")
plt.ylabel("Przewidywane PM2.5")
plt.title(f"Predykcja vs rzeczywistość (R² test = {metrics['r2_test']:.3f})")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "16_ml_pred_vs_true.png", dpi=200)
plt.show()

print("PODSUMOWANIE MODELU")
print("=" * 50)
print(f"Liczba próbek: {metrics['n_samples']}")
print(f"Liczba cech: {metrics['n_features']}")
print(f"R² train: {metrics['r2_train']:.4f}")
print(f"R² test:  {metrics['r2_test']:.4f}")
print(f"RMSE train: {metrics['rmse_train']:.3f}")
print(f"RMSE test:  {metrics['rmse_test']:.3f}")
print(f"MAE test:   {metrics['mae_test']:.3f}")
print("=" * 50)

In [ ]:
# =========================================================
# 28. DASHBOARD REGIONÓW
# =========================================================

region_options = sorted(df[REGION_COL].dropna().unique())
region_dropdown = widgets.Dropdown(
    options=region_options,
    value=region_options[0] if region_options else None,
    description="Region:"
)
region_output = widgets.Output()


def update_region_dashboard(change):
    selected_region = change["new"] if isinstance(change, dict) else change
    df_sub = df[df[REGION_COL] == selected_region].copy()

    with region_output:
        clear_output(wait=True)

        if df_sub.empty:
            print(f"Brak danych dla regionu: {selected_region}")
            return

        country_year = (
            df_sub.groupby([COUNTRY_COL, YEAR_COL])["PM2.5_imputed"]
            .mean()
            .reset_index()
        )

        pivot_country = country_year.pivot(
            index=COUNTRY_COL,
            columns=YEAR_COL,
            values="PM2.5_imputed"
        )

        if not pivot_country.empty:
            fig_heat = px.imshow(
                pivot_country,
                labels=dict(x="Rok", y="Kraj", color="PM2.5"),
                color_continuous_scale="Reds",
                title=f"Średnie PM2.5: kraje w regionie {selected_region}",
                aspect="auto",
                height=max(400, min(len(pivot_country) * 25, 1000)),
            )
            display(fig_heat)

        fig_box = px.box(
            country_year,
            x=YEAR_COL,
            y="PM2.5_imputed",
            points="all",
            hover_name=COUNTRY_COL,
            title=f"Rozkład krajów w regionie {selected_region}",
            template="plotly_white",
        )

        region_avg = df_sub.groupby(YEAR_COL)["PM2.5_imputed"].mean().reset_index()

        fig_box.add_trace(
            go.Scatter(
                x=region_avg[YEAR_COL],
                y=region_avg["PM2.5_imputed"],
                mode="lines+markers",
                name="Średnia regionalna",
                line=dict(color="black", width=3),
                marker=dict(size=9),
            )
        )

        display(fig_box)


if region_options:
    region_dropdown.observe(update_region_dashboard, names="value")
    display(widgets.VBox([region_dropdown, region_output]))
    update_region_dashboard({"new": region_dropdown.value})

In [ ]:
# =========================================================
# 29. DASHBOARD KRAJÓW
# =========================================================

country_options = sorted(df[COUNTRY_COL].dropna().unique())
country_dropdown = widgets.Dropdown(
    options=country_options,
    value="Poland" if "Poland" in country_options else (country_options[0] if country_options else None),
    description="Kraj:"
)
country_output = widgets.Output()


def update_country_dashboard(change):
    selected_country = change["new"] if isinstance(change, dict) else change
    df_sub = df[df[COUNTRY_COL] == selected_country].copy()

    with country_output:
        clear_output(wait=True)

        if df_sub.empty:
            display(Markdown(f"**Brak danych dla:** {selected_country}"))
            return

        pivot_city = df_sub.pivot_table(index=CITY_COL, columns=YEAR_COL, values="PM2.5_imputed")

        if not pivot_city.empty:
            fig_heat = px.imshow(
                pivot_city,
                labels=dict(x="Rok", y="Miasto", color="PM2.5"),
                color_continuous_scale="Reds",
                title=f"Intensywność PM2.5 w miastach: {selected_country}",
                height=max(400, min(len(pivot_city) * 20, 1000)),
            )
            display(fig_heat)

        fig_dist = px.box(
            df_sub,
            x=YEAR_COL,
            y="PM2.5_imputed",
            points="all",
            hover_name=CITY_COL,
            title=f"Rozkład miast w kraju: {selected_country}",
            template="plotly_white",
        )

        country_avg = df_sub.groupby(YEAR_COL)["PM2.5_imputed"].mean().reset_index()
        fig_dist.add_trace(
            go.Scatter(
                x=country_avg[YEAR_COL],
                y=country_avg["PM2.5_imputed"],
                mode="lines+markers",
                name="Średnia krajowa",
                line=dict(color="black", width=3),
                marker=dict(size=10),
            )
        )

        display(fig_dist)


if country_options:
    country_dropdown.observe(update_country_dashboard, names="value")
    display(widgets.VBox([country_dropdown, country_output]))
    update_country_dashboard({"new": country_dropdown.value})

In [ ]:
# =========================================================
# 30. KRÓTKIE WNIOSKI TEKSTOWE 
# =========================================================

pm25_median = df["PM2.5_imputed"].median()
pm25_q90 = df["PM2.5_imputed"].quantile(0.90)
pm25_above_q90_pct = (df["PM2.5_imputed"] >= pm25_q90).mean() * 100

yearly = df.groupby(YEAR_COL)["PM2.5_imputed"].mean()
year_start = int(yearly.index.min())
year_end = int(yearly.index.max())
pm25_start = yearly.loc[year_start]
pm25_end = yearly.loc[year_end]
pm25_change_pct = ((pm25_end - pm25_start) / pm25_start * 100) if pm25_start != 0 else np.nan
trend_word = "wzrost" if pm25_end > pm25_start else "spadek"

top_region = region_summary.index[0]
top_country = country_summary.index[0]
top_city = city_summary.index[0]

display(Markdown(f"""
### Najważniejsze wnioski

- Mediana PM2.5 po imputacji wynosi **{pm25_median:.2f}**.
- Próg 90. percentyla dla PM2.5 to **{pm25_q90:.2f}**.
- W latach **{year_start}-{year_end}** średni poziom PM2.5 zmienił się z **{pm25_start:.2f}** do **{pm25_end:.2f}** ({trend_word} o **{abs(pm25_change_pct):.1f}%**).
- Region z najwyższym średnim PM2.5: **{top_region}**.
- Kraj z najwyższym średnim PM2.5: **{top_country}**.
- Miasto z najwyższym średnim PM2.5: **{top_city}**.
- Model Random Forest osiągnął **R² test = {model_result["metrics"]["r2_test"]:.3f}**.
"""))

In [ ]:
# =========================================================
# CELL 31 — DODATKOWE / ROBOCZE FRAGMENTY
# =========================================================
# Tutaj trafiają rzeczy, które były w notebooku, ale:
# - powtarzały wcześniejsze analizy,
# - były odpalane jednorazowo roboczo,
# - nie są kluczowe dla głównego pipeline'u,
# - albo były pomocnicze przy eksploracji danych.
# =========================================================


# ---------------------------------------------------------
# 1. Szybki podgląd danych i podstawowe statystyki
# ---------------------------------------------------------
print("PODGLĄD DANYCH")
display(df.head())

print("\nINFO O DANYCH")
df.info()

print("\nSTATYSTYKI OPISOWE")
display(df.describe(include="all"))


# ---------------------------------------------------------
# 2. Liczba unikalnych wartości w wybranych kolumnach
# ---------------------------------------------------------
print("\nLICZBA UNIKALNYCH WARTOŚCI")
for col in [REGION_COL, COUNTRY_COL, CITY_COL, YEAR_COL]:
    if col in df.columns:
        print(f"{col}: {df[col].nunique()}")


# ---------------------------------------------------------
# 3. Rozkład braków dla wybranych kolumn
# ---------------------------------------------------------
print("\nBRAKI W KLUCZOWYCH KOLUMNACH")
display(df[[PM25_COL, PM10_COL, NO2_COL, PM25_COV_COL, PM10_COV_COL, NO2_COV_COL]].isna().sum())


# ---------------------------------------------------------
# 4. Dodatkowa tabela: liczba obserwacji na rok
# ---------------------------------------------------------
obs_per_year = df.groupby(YEAR_COL).size().to_frame("n_obs")
print("\nLICZBA OBSERWACJI NA ROK")
display(obs_per_year)


# ---------------------------------------------------------
# 5. Dodatkowa tabela: liczba obserwacji na region
# ---------------------------------------------------------
obs_per_region = df.groupby(REGION_COL).size().sort_values(ascending=False).to_frame("n_obs")
print("\nLICZBA OBSERWACJI NA REGION")
display(obs_per_region)


# ---------------------------------------------------------
# 6. Dodatkowa tabela: liczba obserwacji na kraj
# ---------------------------------------------------------
obs_per_country = df.groupby(COUNTRY_COL).size().sort_values(ascending=False).to_frame("n_obs")
print("\nLICZBA OBSERWACJI NA KRAJ")
display(obs_per_country.head(30))


# ---------------------------------------------------------
# 7. Dodatkowa tabela: liczba obserwacji na miasto
# ---------------------------------------------------------
obs_per_city = df.groupby(CITY_COL).size().sort_values(ascending=False).to_frame("n_obs")
print("\nLICZBA OBSERWACJI NA MIASTO")
display(obs_per_city.head(30))


# ---------------------------------------------------------
# 8. Rozkłady pozostałych zanieczyszczeń
# ---------------------------------------------------------
for pollutant in [PM10_COL, NO2_COL]:
    plt.figure(figsize=(10, 5))
    sns.histplot(df[pollutant].dropna(), bins=40)
    plt.title(f"Rozkład zmiennej: {pollutant}")
    plt.xlabel(pollutant)
    plt.ylabel("Liczba obserwacji")
    plt.tight_layout()
    plt.show()


# ---------------------------------------------------------
# 9. Dodatkowe boxploty dla PM10 i NO2 według regionów
# ---------------------------------------------------------
for pollutant in [PM10_COL, NO2_COL]:
    plt.figure(figsize=(12, 6))
    sns.boxplot(
        data=df,
        x=REGION_COL,
        y=pollutant,
        order=region_order,
        showfliers=False
    )
    plt.title(f"Rozkład {pollutant} w regionach")
    plt.xlabel(REGION_COL)
    plt.ylabel(pollutant)
    plt.xticks(rotation=40, ha="right")
    plt.tight_layout()
    plt.show()


# ---------------------------------------------------------
# 10. Dodatkowa macierz współczynników korelacji Pearsona dla coverage
# ---------------------------------------------------------
coverage_cols = [col for col in [PM25_COV_COL, PM10_COV_COL, NO2_COV_COL] if col in df.columns]
if coverage_cols:
    corr_cov = df[coverage_cols].corr()

    plt.figure(figsize=(6, 5))
    sns.heatmap(corr_cov, annot=True, fmt=".2f", cmap="Blues")
    plt.title("Macierz współczynników korelacji Pearsona dla coverage")
    plt.tight_layout()
    plt.show()


# ---------------------------------------------------------
# 11. Średnie poziomy PM10 i NO2 według regionów
# ---------------------------------------------------------
region_other_pollutants = (
    df.groupby(REGION_COL)
    .agg(
        pm10_mean=(PM10_COL, "mean"),
        no2_mean=(NO2_COL, "mean")
    )
    .sort_values("pm10_mean", ascending=False)
)

print("\nŚREDNIE PM10 I NO2 WG REGIONÓW")
display(region_other_pollutants)


# ---------------------------------------------------------
# 12. Średnie poziomy PM10 i NO2 według krajów
# ---------------------------------------------------------
country_other_pollutants = (
    df.groupby(COUNTRY_COL)
    .agg(
        pm10_mean=(PM10_COL, "mean"),
        no2_mean=(NO2_COL, "mean")
    )
    .sort_values("pm10_mean", ascending=False)
)

print("\nŚREDNIE PM10 I NO2 WG KRAJÓW")
display(country_other_pollutants.head(20))


# ---------------------------------------------------------
# 13. Średnie poziomy PM10 i NO2 według miast
# ---------------------------------------------------------
city_other_pollutants = (
    df.groupby(CITY_COL)
    .agg(
        pm10_mean=(PM10_COL, "mean"),
        no2_mean=(NO2_COL, "mean")
    )
    .sort_values("pm10_mean", ascending=False)
)

print("\nŚREDNIE PM10 I NO2 WG MIAST")
display(city_other_pollutants.head(20))


# ---------------------------------------------------------
# 14. Dodatkowy trend globalny PM10 i NO2 w czasie
# ---------------------------------------------------------
global_year_other = (
    df.groupby(YEAR_COL)
    .agg(
        pm10_mean=(PM10_COL, "mean"),
        no2_mean=(NO2_COL, "mean")
    )
    .reset_index()
)

plt.figure(figsize=(11, 5))
plt.plot(global_year_other[YEAR_COL], global_year_other["pm10_mean"], marker="o", label="PM10")
plt.plot(global_year_other[YEAR_COL], global_year_other["no2_mean"], marker="o", label="NO2")
plt.title("Globalny trend PM10 i NO2 w czasie")
plt.xlabel("Rok")
plt.ylabel("Średnia wartość")
plt.legend()
plt.tight_layout()
plt.show()


# ---------------------------------------------------------
# 15. Robocza lista lat
# ---------------------------------------------------------
years = sorted(df[YEAR_COL].dropna().unique())
print("\nDOSTĘPNE LATA:")
print(years)


# ---------------------------------------------------------
# 16. Dodatkowe sprawdzenie udziału imputacji według krajów
# ---------------------------------------------------------
imputation_by_country = (
    df.groupby(COUNTRY_COL)["PM2.5_was_imputed"]
    .mean()
    .mul(100)
    .round(2)
    .sort_values(ascending=False)
    .to_frame("udzial_imputowanych_%")
)

print("\nUDZIAŁ IMPUTOWANYCH OBSERWACJI WG KRAJÓW")
display(imputation_by_country.head(20))


# ---------------------------------------------------------
# 17. Dodatkowe sprawdzenie udziału imputacji według miast
# ---------------------------------------------------------
imputation_by_city = (
    df.groupby(CITY_COL)["PM2.5_was_imputed"]
    .mean()
    .mul(100)
    .round(2)
    .sort_values(ascending=False)
    .to_frame("udzial_imputowanych_%")
)

print("\nUDZIAŁ IMPUTOWANYCH OBSERWACJI WG MIAST")
display(imputation_by_city.head(20))


# ---------------------------------------------------------
# 18. Robocze porównania dla pojedynczych krajów
# ---------------------------------------------------------
selected_countries = ["China", "India", "Poland", "Italy", "United States of America"]

for country in selected_countries:
    if country in df[COUNTRY_COL].unique():
        temp = df[df[COUNTRY_COL] == country]
        yearly_country = temp.groupby(YEAR_COL)["PM2.5_imputed"].mean().reset_index()

        plt.figure(figsize=(8, 4))
        plt.plot(yearly_country[YEAR_COL], yearly_country["PM2.5_imputed"], marker="o")
        plt.title(f"Trend PM2.5 dla kraju: {country}")
        plt.xlabel("Rok")
        plt.ylabel("PM2.5")
        plt.tight_layout()
        plt.show()


# ---------------------------------------------------------
# 19. Robocze porównania dla pojedynczych regionów
# ---------------------------------------------------------
for region in region_order:
    temp = df[df[REGION_COL] == region]
    yearly_region = temp.groupby(YEAR_COL)["PM2.5_imputed"].mean().reset_index()

    plt.figure(figsize=(8, 4))
    plt.plot(yearly_region[YEAR_COL], yearly_region["PM2.5_imputed"], marker="o")
    plt.title(f"Trend PM2.5 dla regionu: {region}")
    plt.xlabel("Rok")
    plt.ylabel("PM2.5")
    plt.tight_layout()
    plt.show()

# ---------------------------------------------------------
# 20. Robocze top listy dla PM10 i NO2
# ---------------------------------------------------------
top_pm10_countries = (
    df.groupby(COUNTRY_COL)[PM10_COL]
    .median()
    .sort_values(ascending=False)
    .head(15)
    .to_frame("PM10_median")
)

top_no2_countries = (
    df.groupby(COUNTRY_COL)[NO2_COL]
    .median()
    .sort_values(ascending=False)
    .head(15)
    .to_frame("NO2_median")
)

print("\nTOP KRAJE WG MEDIANY PM10")
display(top_pm10_countries)

print("\nTOP KRAJE WG MEDIANY NO2")
display(top_no2_countries)